# LangGraphDeep Dive

**Module 11 · Lesson 11.3**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- A Graph That Executes: StateGraph
- State & Reducers: the Merge Problem
- Conditional Edges & Routing
- Cycles: the Agent Loop as a Graph
- Persistence & Checkpointing
- Parallelism: Fan-Out & the Send API

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install langchain langchain-anthropic langchain-core langgraph python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## The while-loop hits a wall. The graph does not.

> 💡 **Info**
>
> Your 11.1 agent was a `while` loop with `if` statements. It works - until production asks four things a loop cannot do cleanly: **save and resume** after a crash, **branch** by topic, run steps **in parallel**, and **pause for a human**. LangGraph answers all four with one idea: model the agent as a **graph**. Nodes are functions, edges are transitions, and a shared typed **state** flows through - and because it is a graph, the runtime can checkpoint every step, route conditionally, fan out, and pause. This lesson builds that graph from the ground up, one primitive at a time, and every example is *real LangGraph you can run* - no model, no key, just the machinery.

### 🎯 What you will be able to do after this lesson

- **Build** a StateGraph from typed state, nodes, and edges, and run it.

- **Compare** reducer behaviors (`operator.add` vs `add_messages` vs last-write-wins) and see why a parallel write with no reducer raises `InvalidUpdateError`.

- **Operate** persistence (thread_id, resume, time-travel) and parallelism (fan-out and the `Send` API).

- **Evaluate** when to build from scratch vs use `create_agent`, and pick the right streaming mode and checkpoint backend.

> 📦 **Info**
>
> ✅ Before you startThis builds on **11.1** (the observe-think-act loop; cap iterations because reliability compounds) and **11.2** (ReAct and routing as architectures). You should be comfortable reading a short Python class and a `TypedDict`. This lesson is LangGraph **mechanics**; deep human-in-the-loop is Lesson 11.10, memory systems are 11.4, and multi-agent is 11.5.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🗺️ **Analogy**
>
> A hand-rolled agent loop is **directions you recite from memory**; LangGraph is a **GPS with a saved map**. The map is a graph of roads (edges) and junctions (nodes); your position and cargo are the **state**; the GPS can reroute at a junction (conditional edge), take two roads at once (parallel), and - because it saved your route - resume after you stop for fuel (checkpointing). **Where it breaks down:** a graph is more moving parts than a loop, so for a truly simple one-shot task, a plain call is still better (11.2's ladder).

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“A node edits the state object.”** Nodes return a *partial update dict*; the graph merges it using **reducers**. Mutating state directly bypasses checkpointing and corrupts it.
> - **“`operator.add` and `add_messages` are the same for chat.”** `operator.add` blindly concatenates; `add_messages` dedups by message id (so an edited message replaces the original). Use `add_messages` for the `messages` key.

> 💡 **Info**
>
> ⚠️ Anti-patternReaching for LangGraph on a task that a single call or a fixed chain would handle (11.2’s ladder). The graph earns its complexity only when you need persistence, branching, parallelism, or human checkpoints. Start at the lowest rung; climb to a graph when one of those forces it.

---

## 🎯 Concept 1: A Graph That Executes: StateGraph

### A Graph That Executes: StateGraph

Typed state flows through nodes along edges. Compile once, invoke many times. No LLM needed to see the machine.

LangGraph’s core object is the **StateGraph**. You define a **state** (a `TypedDict` every node shares), add **nodes** (each a function that reads the state and returns a *partial update*), wire **edges** between them (starting at `START`, ending at `END`), then `compile()` it into a runnable app you `invoke()`. That is the whole model: a flowchart that executes. Below, two plain-function nodes - `parse` then `respond` - run in sequence with no model at all, so you can watch the state and the path flow through.

> 🏭 **Analogy**
>
> It’s an **assembly line**. The part on the belt is the **state**; each station (a **node**) does one operation and passes the part along the belt (an **edge**) to the next. `START` is where the part enters, `END` is the shipping dock. Compiling the graph is bolting the line together; invoking it is switching the belt on.

A node returns `{"answer": "14999 INR"}` but the state has three keys. What happens to the other two keys?

**📝 `01_stategraph.py`** - *LangGraph*

In [ ]:
# A STATEGRAPH IS A FLOWCHART THAT EXECUTES: typed State flows through Nodes along Edges.
# No LLM here - plain functions - so you see the machine, not the model.
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):          # the shared state every node reads and writes
    query: str
    path: list
    answer: str

def parse(state):                # a NODE = a function of state -> a PARTIAL state update
    return {"query": state["query"].strip().lower(), "path": state["path"] + ["parse"]}

def respond(state):
    price = {"genai": 14999, "python": 9999}.get(state["query"], 0)
    return {"answer": f"{price} INR", "path": state["path"] + ["respond"]}

graph = StateGraph(State)
graph.add_node("parse", parse)
graph.add_node("respond", respond)
graph.add_edge(START, "parse")      # EDGES wire the flow: START -> parse -> respond -> END
graph.add_edge("parse", "respond")
graph.add_edge("respond", END)
app = graph.compile()               # compile once, invoke many times

out = app.invoke({"query": "  GenAI  ", "path": [], "answer": ""})
print("path:  ", " -> ".join(["START"] + out["path"] + ["END"]))
print("answer:", out["answer"])

# Output:
# path:   START -> parse -> respond -> END
# answer: 14999 INR

- `State` is a `TypedDict` - the shared shape every node reads and writes.
- Each node returns a **partial** dict; unmentioned keys are left as they were.
- Edges wire the order: `START → parse → respond → END`, exactly the path printed.
- `compile()` builds the runnable app; `invoke()` runs one pass and returns the final state.

#### 💡 What just happened

⚡ What just happened? A StateGraph is **typed state + nodes + edges**, compiled and invoked. Tradeoff / when it matters: this is more ceremony than a `while` loop, and it only pays off once you need what a graph adds (next six steps). For a one-shot task, a plain call still wins.

- Step the graph START -> parse -> respond -> END
- Watch the typed state object update at each node

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: State & Reducers: the Merge Problem

### State & Reducers: the Merge Problem

When two nodes update the same key, a reducer decides how. This is THE core LangGraph concept.

Here is the concept that trips up every beginner. Nodes return partial updates, and the graph **merges** them into the state. But what if two nodes update the *same* key - say, two parallel branches both appending to a list? The **reducer** annotated on that key decides the merge. Three behaviors matter: **no reducer** = last-write-wins (fine for a single-value field like `route`, but two writes in one step *crash* with `InvalidUpdateError`); **`operator.add`** = concatenate lists (accumulate results); and **`add_messages`** = the chat-aware reducer that appends but *dedups by message id*, so re-sending a message with the same id *replaces* it instead of duplicating. Get this wrong and parallel graphs crash or silently lose data; get it right and everything downstream (parallelism, HITL editing) just works.

> 🪛 **Analogy**
>
> State is a **shared whiteboard** and reducers are the **rule for when two people write at once**. ‘Last one wins’ (no reducer) is fine for a single sticky note but chaos if both write together. ‘Add both’ (`operator.add`) keeps every entry. ‘Match by name-tag’ (`add_messages`) replaces your old note when you rewrite it, instead of leaving two. Pick the wrong rule and the board is either a crash or a mess.

Two parallel nodes both do `return {"out": [...]}` on a key with **no** reducer. What happens?

**📝 `02_reducers.py`** - *LangGraph*

In [ ]:
# THE MERGE PROBLEM: when two nodes update the SAME key in one step, who wins? A REDUCER decides.
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.errors import InvalidUpdateError

def run_two_parallel(state_type, seed):
    def a(s): return {"out": ["from_a"]}
    def b(s): return {"out": ["from_b"]}
    g = StateGraph(state_type)
    g.add_node("a", a); g.add_node("b", b)
    g.add_edge(START, "a"); g.add_edge(START, "b")   # a and b run IN PARALLEL (same step)
    g.add_edge("a", END); g.add_edge("b", END)
    return g.compile().invoke(seed)

class WithReducer(TypedDict):
    out: Annotated[list, operator.add]        # operator.add: concatenate both updates

print("with operator.add ->", run_two_parallel(WithReducer, {"out": []})["out"])

class NoReducer(TypedDict):
    out: list                                 # no reducer: two writes in one step = crash

try:
    run_two_parallel(NoReducer, {"out": []})
except InvalidUpdateError:
    print("without a reducer ->", "InvalidUpdateError: can receive only one value per step")

# Output:
# with operator.add -> ['from_a', 'from_b']
# without a reducer -> InvalidUpdateError: can receive only one value per step

- With `operator.add` the two parallel writes concatenate to `['from_a', 'from_b']`.
- With **no** reducer the same graph raises `InvalidUpdateError` - the merge is undefined.
- The fix is always to annotate the shared key with a reducer that defines the merge.
- Single-value keys (like `route`) need no reducer - last-write-wins is what you want there.

**📝 `02b_add_messages.py`** - *LangGraph*

In [ ]:
# add_messages: the reducer built for CHAT. It appends new messages, but a message whose id
# already exists REPLACES the old one (dedup) - which is what makes human message-editing work.
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

base = [HumanMessage(content="hi", id="1"), AIMessage(content="hello", id="2")]
appended = add_messages(base, [AIMessage(content="how can I help?", id="3")])  # new id -> append
edited   = add_messages(base, [HumanMessage(content="hi (edited)", id="1")])   # same id -> replace
print("append (new id 3):", [m.content for m in appended])
print("edit   (reuse id 1):", [m.content for m in edited])

# Output:
# append (new id 3): ['hi', 'hello', 'how can I help?']
# edit   (reuse id 1): ['hi (edited)', 'hello']

- `add_messages` appends a message with a new id (id `3` lands at the end).
- A message re-sent with an *existing* id (id `1`) **replaces** the old one in place.
- That dedup-by-id is exactly what lets a human edit a past message without duplicating it.
- Always use `add_messages` for the `messages` key; never `operator.add`.

#### 💡 What just happened

⚡ What just happened? Reducers are the merge rule for shared state: `operator.add` concatenates, `add_messages` dedups by id, no-reducer is last-write-wins (and crashes on a parallel write). Tradeoff: you must think about the merge on every shared key - but that is exactly what makes parallelism and human editing safe. This is the concept the rest of LangGraph rests on.

- Two parallel nodes write one key; flip the reducer
- See none -> crash, operator.add -> merge, add_messages -> dedup by id

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Conditional Edges & Routing

### Conditional Edges & Routing

A routing function reads state and names the next node. Branch by topic, all inside the graph.

A plain edge always goes to the same next node. A **conditional edge** asks a function which way to go. You attach it with `add_conditional_edges(source, routing_fn, mapping)`: the routing function reads the state and returns a key (here, a route name), and the mapping sends that key to a node. This is 11.2’s *routing* pattern expressed as a graph - a cheap classifier node decides `billing` / `technical` / `general`, and the conditional edge lights exactly one downstream branch. No LLM needed to see it work.

> 🚄 **Analogy**
>
> It’s a **railway switch** (the points). A plain edge is a track that only goes one way; a conditional edge is the **switch operator** who reads the train’s destination card and throws the lever to send it down the right line. The routing function is that operator; the mapping is the set of levers.

The routing function returns `"technical"`. How does the graph use that string?

**📝 `03_conditional_routing.py`** - *LangGraph*

In [ ]:
# CONDITIONAL EDGES: a routing function reads state and NAMES the next node. Branch as a graph.
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    query: str
    route: str
    answer: str

def classify(state):                              # a cheap keyword router (no LLM)
    q = state["query"].lower()
    if any(w in q for w in ("refund", "price", "emi")): return {"route": "billing"}
    if any(w in q for w in ("python", "module", "prereq")): return {"route": "technical"}
    return {"route": "general"}

def billing(state):   return {"answer": "[billing] 7-day full refund; GenAI 14999 INR"}
def technical(state): return {"answer": "[technical] prereqs: Python + HS math; 14 modules"}
def general(state):   return {"answer": "[general] how can I help?"}

def route_decision(state): return state["route"]  # returns the NAME of the next node

g = StateGraph(State)
for name, fn in [("classify", classify), ("billing", billing),
                 ("technical", technical), ("general", general)]:
    g.add_node(name, fn)
g.add_edge(START, "classify")
g.add_conditional_edges("classify", route_decision,        # map route -> node
                        {"billing": "billing", "technical": "technical", "general": "general"})
for n in ("billing", "technical", "general"): g.add_edge(n, END)
app = g.compile()

for q in ["Can I get a refund?", "What Python version?", "Hello!"]:
    out = app.invoke({"query": q, "route": "", "answer": ""})
    print(f"{q:24s} -> {out['route']:9s} -> {out['answer']}")

# Output:
# Can I get a refund?      -> billing   -> [billing] 7-day full refund; GenAI 14999 INR
# What Python version?     -> technical -> [technical] prereqs: Python + HS math; 14 modules
# Hello!                   -> general   -> [general] how can I help?

- `classify` is a plain keyword router - it writes a `route` into the state.
- `add_conditional_edges` calls `route_decision`, which returns the route name.
- The mapping turns that name into the next node (billing / technical / general).
- Each query lights exactly one branch - the same routing pattern from 11.2, now as a graph.
- Modern shortcut (LangGraph 1.0): a node can `return Command(update={...}, goto="next")` to update state AND route in one step, instead of a plain dict plus a separate conditional edge.

#### 💡 What just happened

⚡ What just happened? A conditional edge is a function that reads state and names the next node; it is how a graph branches. Tradeoff: the routing function’s accuracy caps the whole graph (a misroute sends the query to the wrong specialist), so invest in it - exactly the routing lesson from 11.2.

- Send a query and watch the routing function pick a branch
- The chosen edge to billing / technical / general lights up

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Cycles: the Agent Loop as a Graph

### Cycles: the Agent Loop as a Graph

The agent-tools loop is a cycle: agent routes to tools, tools route back. A recursion_limit keeps it safe.

Every graph so far ran forward to `END`. An agent needs a **cycle**: call the model, and if it asked for a tool, run the tool and go *back* to the model. In LangGraph that is a conditional edge (`should_continue`) that routes the agent either to the `tools` node or to `END`, plus a plain edge from `tools` back to `agent`. That is the exact observe-think-act loop from 11.1, now drawn as a graph - the agent node is scripted here so it runs with no key. And because a cycle *could* loop forever, LangGraph enforces a **`recursion_limit`** (default twenty-five): the same “cap your iterations” rule that 11.1’s `p^N` reliability argument demanded, now built into the runtime.

> 🍳 **Analogy**
>
> It’s a **cook tasting and re-seasoning**: taste (agent), decide it needs salt (route to tools), add salt (tools), taste again (back to agent) - a loop that ends when the dish is right. The **kitchen timer** is the `recursion_limit`: even a cook who never stops seasoning is forced to plate when it rings, so nothing runs forever.

The agent node returns a tool call. Which edge fires, and where does the graph go after the tool runs?

**📝 `04_agent_loop.py`** - *LangGraph*

In [ ]:
# THE AGENT LOOP AS A GRAPH: agent -> (tool_calls?) -> tools -> back to agent, until done.
# The agent node is SCRIPTED so this runs with no key, but the graph is real LangGraph.
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    task: str
    scratch: Annotated[list, operator.add]   # accumulates think/act/observe lines
    answer: str

KB = {"genai": "14999"}
def tool(name, arg):
    if name == "search":    return KB.get(arg, "not found")
    if name == "calculate": return str(round(eval(arg, {"__builtins__": {}}), 2))
    return "unknown tool"

def agent(state):                            # SCRIPTED think step (a real LLM would decide this)
    acts = [s for s in state["scratch"] if s.startswith("act")]
    if not any("search" in a for a in acts):
        return {"scratch": ["think: need the price", "act: search(genai)"]}
    if not any("calculate" in a for a in acts):
        return {"scratch": ["think: add 18% GST", "act: calculate(14999*1.18)"]}
    return {"answer": "17698.82 INR", "scratch": ["think: done"]}

def tools(state):
    last = state["scratch"][-1]              # e.g. "act: search(genai)"
    call = last.split("act:")[1].strip()
    name, arg = call[:-1].split("(")
    return {"scratch": [f"observe: {tool(name, arg)}"]}

def should_continue(state):                  # conditional edge: loop or stop
    return "tools" if state["scratch"][-1].startswith("act") else "end"

g = StateGraph(State)
g.add_node("agent", agent); g.add_node("tools", tools)
g.add_edge(START, "agent")
g.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
g.add_edge("tools", "agent")                 # the CYCLE: tools -> agent
app = g.compile()

out = app.invoke({"task": "GenAI price with 18% GST?", "scratch": [], "answer": ""},
                 {"recursion_limit": 12})    # a hard cap so a stuck graph cannot spin forever
for line in out["scratch"]: print(" ", line)
print("answer:", out["answer"])

# Output:
#   think: need the price
#   act: search(genai)
#   observe: 14999
#   think: add 18% GST
#   act: calculate(14999*1.18)
#   observe: 17698.82
#   think: done
# answer: 17698.82 INR

- `should_continue` is the conditional edge: tool call → `tools`, otherwise → `END`.
- The plain edge `tools → agent` closes the cycle, so the model sees each observation.
- The trace is the same think/act/observe loop from 11.1, now executed by the graph.
- `recursion_limit` is the hard cap - a graph that never answers is stopped, not left spinning.
- In real code you rarely hand-roll these two: `ToolNode(tools)` (from `langgraph.prebuilt`) executes the model’s `.tool_calls` for you, and `tools_condition` is the prebuilt `should_continue` - the scripted versions here just keep the demo keyless.

#### 💡 What just happened

⚡ What just happened? An agent is a **cyclic graph**: agent routes to tools, tools route back, until the agent stops asking for tools. Tradeoff / safety: a cycle can loop forever, so the `recursion_limit` (default 25) enforces 11.1’s cap-your-iterations rule at the runtime level. You still design a real termination condition; the limit is the backstop.

- Watch agent -> tools -> agent cycle think / act / observe
- should_continue routes back; a recursion_limit counter ticks

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Persistence & Checkpointing

### Persistence & Checkpointing

Compile with a checkpointer + a thread_id and state survives between calls: memory, resume, time-travel.

This is LangGraph’s killer feature. Compile the graph with a **checkpointer** and pass a **`thread_id`**, and the runtime *autosaves the full state after every node*. Three things fall out for free. **Memory:** invoke the same thread again and the prior messages are still there (turn two understands “it” from turn one). **Resume:** if the process crashes, the last checkpoint is on disk - restart and continue. **Time-travel:** `get_state_history` returns every checkpoint, so you can inspect or fork from any past step instead of replaying the whole run. Use `InMemorySaver` for dev, `SqliteSaver` (SQLite) for local, and `PostgresSaver` (PostgreSQL) for production; a Redis-backed saver covers high-throughput sessions. (Checkpointing is also what makes `interrupt()` - pausing for a human - possible; the deep human-in-the-loop treatment is Lesson 11.10.)

> 💾 **Analogy**
>
> It’s a **video-game autosave**. The checkpointer saves your game after every room (node); the `thread_id` is which **save slot** you are in. Reload a slot to continue where you left off (memory + resume), or load an *earlier* save to replay a tricky room differently (time-travel). A brand-new slot starts you at the beginning with nothing.

You invoke a checkpointed graph twice with the same `thread_id`. On the second call, what is in the state?

**📝 `05_persistence.py`** - *LangGraph*

In [ ]:
# PERSISTENCE: compile with a checkpointer + pass a thread_id, and state SURVIVES between calls.
# This is how a graph gets memory, resume-after-crash, and time-travel - for free.
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver   # swap for SqliteSaver / PostgresSaver in prod
from langchain_core.messages import HumanMessage, AIMessage

class State(TypedDict):
    messages: Annotated[list, add_messages]

def turn(state):                             # canned reply (no LLM) so the run is deterministic
    user = state["messages"][-1].content
    reply = "GenAI course is 14999 INR." if "price" in user.lower() else "With 18% GST: 17698.82 INR."
    return {"messages": [AIMessage(content=reply)]}

g = StateGraph(State); g.add_node("turn", turn)
g.add_edge(START, "turn"); g.add_edge("turn", END)
app = g.compile(checkpointer=InMemorySaver())            # <-- the one line that adds memory

cfg = {"configurable": {"thread_id": "user-123"}}
app.invoke({"messages": [HumanMessage(content="GenAI price?")]}, cfg)   # turn 1
app.invoke({"messages": [HumanMessage(content="With 18% GST?")]}, cfg)  # turn 2, SAME thread

state = app.get_state(cfg)
print("thread user-123 remembers", len(state.values["messages"]), "messages:")
for m in state.values["messages"]: print("   ", type(m).__name__, "-", m.content)
print("checkpoints saved (time-travel):", len(list(app.get_state_history(cfg))))

fresh = app.get_state({"configurable": {"thread_id": "user-999"}})
print("a NEW thread starts empty:", fresh.values.get("messages", []))

# Output:
# thread user-123 remembers 4 messages:
#     HumanMessage - GenAI price?
#     AIMessage - GenAI course is 14999 INR.
#     HumanMessage - With 18% GST?
#     AIMessage - With 18% GST: 17698.82 INR.
# checkpoints saved (time-travel): 6
# a NEW thread starts empty: []

- Two invokes on `thread_id user-123` leave **four** messages - the thread remembered turn one.
- `get_state_history` returns every checkpoint saved (here six across the two turns) - each time-travel point you can inspect or fork from.
- The count is a few per invoke, not literally one per node: LangGraph checkpoints each super-step (the input and each step of the run), which is why two turns give six.
- A different `thread_id` (`user-999`) starts empty - threads are isolated; swap `InMemorySaver` for `SqliteSaver`/`PostgresSaver` and the state survives restarts.

#### 💡 What just happened

⚡ What just happened? A checkpointer + `thread_id` gives you persistence: multi-turn memory, resume-after-crash, and time-travel, for one line at compile time. Tradeoff / scope: this is short-term *thread* state, not long-term cross-session memory (that is Lesson 11.4), and it is the mechanism behind human-in-the-loop pauses (Lesson 11.10).

- Checkpoints save at each node; scrub the timeline to time-travel
- Two threads side by side: one remembers, a new one is empty

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Parallelism: Fan-Out & the Send API

### Parallelism: Fan-Out & the Send API

Multiple edges out of a node run in parallel. Send() fans out a dynamic number of workers, then a reducer fans them in.

Because the graph knows the whole structure, it can run independent nodes **at the same time**. Two ways. **Static fan-out:** draw two edges out of one node and both destinations run in the same step; a downstream node with two incoming edges *waits for both* (fan-in), and a reducer merges what they produced. **Dynamic fan-out (the `Send` API):** when you do not know how many parallel tasks you need until runtime - one per document, per query, per row - a routing function returns a list of `Send("node", state)` objects, and LangGraph launches one worker per `Send`. This is clean map-reduce: map with `Send`, reduce with a reducer on the shared key. It is the parallelization pattern from 11.2, now with real fan-in.

> 🍳 **Analogy**
>
> It’s a **kitchen at dinner rush**. The expediter reads a big order and hands one ticket to each line cook at once (fan-out / `Send`); the cooks work in parallel; the pass **plates the dish only when every ticket is done** (fan-in), combining their parts (the reducer). If you do not know how many dishes until the order comes in, `Send` makes exactly that many tickets.

A `Send`-based map-reduce runs over 3 documents. How many worker instances run, and how do results combine?

**📝 `06_parallel_send.py`** - *LangGraph*

In [ ]:
# PARALLELISM: multiple edges out of a node run IN PARALLEL; Send() fans out DYNAMICALLY.
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# -- Static fan-out / fan-in: two nodes run together, a reducer merges their results --
class FanState(TypedDict):
    facts: Annotated[list, operator.add]
def price_lookup(s):  return {"facts": ["price: 14999"]}
def hours_lookup(s):  return {"facts": ["hours: 146"]}
def summarize(s):     return {"facts": ["summary: " + " | ".join(sorted(s["facts"]))]}
g = StateGraph(FanState)
g.add_node("price", price_lookup); g.add_node("hours", hours_lookup); g.add_node("sum", summarize)
g.add_edge(START, "price"); g.add_edge(START, "hours")   # fan-out
g.add_edge("price", "sum");  g.add_edge("hours", "sum")   # fan-in (sum waits for BOTH)
g.add_edge("sum", END)
print("fan-out/fan-in:", g.compile().invoke({"facts": []})["facts"][-1])

# -- Dynamic map-reduce with Send: number of workers unknown until runtime --
class MapState(TypedDict):
    docs: list
    scored: Annotated[list, operator.add]
def fan_out(state):                          # returns one Send per doc -> N parallel workers
    return [Send("score", {"doc": d}) for d in state["docs"]]
class WorkerState(TypedDict):
    doc: str
    scored: Annotated[list, operator.add]
def score(state): return {"scored": [f"{state['doc']}:{len(state['doc'])}"]}
m = StateGraph(MapState)
m.add_node("score", score)
# NOTE: unlike Step 3's DICT mapping, the 3rd arg here is just the LIST of possible
# target nodes - fan_out returns Send objects directly, not a route key to look up.
m.add_conditional_edges(START, fan_out, ["score"])
m.add_edge("score", END)
print("Send map-reduce:", sorted(m.compile().invoke({"docs": ["genai", "python", "sql"], "scored": []})["scored"]))

# Output:
# fan-out/fan-in: summary: hours: 146 | price: 14999
# Send map-reduce: ['genai:5', 'python:6', 'sql:3']

- Static: `price` and `hours` fan out, run together, and `sum` fans them in via the reducer.
- Dynamic: `fan_out` returns one `Send` per doc, so the worker count is decided at runtime.
- Each worker writes to a reducer-annotated key, so their results merge instead of clashing.
- Note the third argument here is a *list* of target nodes (not Step 3’s dict) - `fan_out` returns `Send` objects directly, so there is no route key to map.
- This is 11.2’s parallelization pattern with real fan-in - and a reducer is mandatory on the shared key.

#### 💡 What just happened

⚡ What just happened? Parallelism comes from graph structure: multiple edges run in parallel, and `Send` fans out a runtime-decided number of workers (map-reduce). Tradeoff: every shared key a parallel branch writes MUST have a reducer, or you get the `InvalidUpdateError` from Step 2. Structure gives you speed; reducers keep it safe.

- Slide the document count and watch Send fan out N workers
- A reducer fans the parallel results back in

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Prebuilt & Production: create_agent, Streaming, Observability

### Prebuilt & Production: create_agent, Streaming, Observability

One line builds the ReAct graph. Stream for UX; trace with LangSmith. Build from scratch only when you need to.

You have now built by hand every piece the standard agent needs, so the shortcut will make sense. LangChain’s **`create_agent(model, tools)`** (from `langchain.agents` - the 2026 successor to `langgraph.prebuilt.create_react_agent`) builds the entire agent→tools→agent graph in one call, compiling to the same StateGraph you just wrote. Two production essentials come with it. **Streaming:** `app.stream(..., stream_mode="updates")` yields a state delta per node (cheapest, best for UIs); `"messages"` streams tokens for a live-typing effect; `"values"` streams the full state. **Observability:** set `LANGSMITH_TRACING=true` and every node run is traced in LangSmith and visualized in Studio, for free. Reach past the one-liner and build from scratch only when you need custom nodes, non-ReAct routing, or subgraphs - which is exactly why you learned the primitives first.

> 🍲 **Analogy**
>
> `create_agent` is a **meal kit**: the standard recipe (a ReAct agent), portioned and ready, on the table fast. Building the StateGraph by hand is **cooking from scratch** - slower, but the only way when you want a dish the kit does not offer (a custom node, an odd route, a subgraph). You learned to cook first, so the kit is a convenience, not a black box.

You want a live typing effect - tokens appearing as the model generates. Which stream mode?

**📝 `07_prebuilt.py`** - *illustrative*

**`07_prebuilt.py`** *(illustrative - needs a live model + key; not run in this keyless notebook)*

```python
# THE ONE-LINER: create_agent builds the whole agent -> tools graph for you (LangGraph underneath).
from langchain.agents import create_agent      # 2026 entry point (was langgraph.prebuilt.create_react_agent)
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool

@tool
def search(q: str) -> str:
    "Look up a Netsetos course by name."
    return {"genai": "14999 INR"}.get(q.lower(), "not found")

agent = create_agent(ChatAnthropic(model="claude-sonnet-4-6"), tools=[search])

# Stream the run so users see progress instead of a long spinner:
for chunk in agent.stream(
        {"messages": [{"role": "user", "content": "GenAI course price?"}]},
        stream_mode="updates"):            # "updates" = state delta per node (cheap, best for UIs)
    print(chunk)                           # other modes: "messages" (tokens), "values" (full state)

# Output: (illustrative minimal example - needs `pip install langchain langchain-anthropic` + a key.)
#   create_agent replaces ~15 lines of StateGraph wiring with one call; it still compiles to the same
#   agent -> tools -> agent graph you built by hand, so you can add nodes or swap the checkpointer later.
#   Observability is free: set LANGSMITH_TRACING=true and every node run is traced in LangSmith / Studio.
```

- `create_agent` builds the whole agent→tools graph in one line - the same graph you wired by hand.
- It is the 2026 entry point; `langgraph.prebuilt.create_react_agent` is the deprecated predecessor.
- `stream_mode` picks the granularity: `updates` (per node), `messages` (tokens), `values` (full state).
- Observability is a free env var: `LANGSMITH_TRACING=true` traces every node in LangSmith / Studio.

#### 💡 What just happened

⚡ What just happened?`create_agent` is the one-liner for the standard ReAct graph; streaming and LangSmith tracing are the production essentials that come with it. Tradeoff / when to build from scratch: the one-liner only expresses the ReAct loop - the moment you need custom nodes, routing, or subgraphs, drop to the StateGraph primitives you learned in Steps 1–6.

- Toggle build-from-scratch (7 nodes) vs create_agent (1 line)
- Pick a stream mode and watch updates / messages / values chunks flow

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> An agent is a **graph**: typed **state** through **nodes** along **edges**, compiled and invoked (Step 1). When updates collide, a **reducer** decides the merge - `operator.add`, `add_messages`, or last-write-wins - and a parallel write with no reducer raises `InvalidUpdateError` (Step 2). A **conditional edge** reads state and names the next node, so the graph branches (Step 3). Route a node back to itself and you have a **cycle** - the agent loop - capped by `recursion_limit` (Step 4). A **checkpointer** plus a `thread_id` gives memory, resume, and time-travel (Step 5). Draw parallel edges or use the **`Send`** API and the graph fans out, reducers fanning results back in (Step 6). And **`create_agent`** packages the standard ReAct graph in one line, with streaming and LangSmith tracing for production (Step 7). The graph model is the difference between a demo loop and a production agent.

| LangGraph primitive | What it is | Why a while-loop cannot do it cleanly |
|---|---|---|
| Reducer | merge rule for a shared state key | a loop has no safe merge for concurrent updates |
| Conditional edge | function that names the next node | branching by state, declaratively |
| Checkpointer | autosave state per node, per thread | resume-after-crash and time-travel |
| Send API | dynamic parallel fan-out | runtime-width map-reduce with fan-in |
| Command | node-return that updates state and routes | merge an update and agotoin one node |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now build production agents as graphs. Next: **memory systems** (episodic and long-term stores beyond thread state) in Lesson 11.4; **multi-agent** coordination - supervisor, swarm, handoffs - in Lesson 11.5; and deep **human-in-the-loop** with `interrupt()` and approval workflows (which checkpointing makes possible) in Lesson 11.10. The reference is the [LangGraph low-level concepts](https://langchain-ai.github.io/langgraph/concepts/low_level/) and [persistence](https://langchain-ai.github.io/langgraph/concepts/persistence/) docs, the [LangGraph source on GitHub](https://github.com/langchain-ai/langgraph), and Anthropic’s [Building Effective Agents](https://www.anthropic.com/research/building-effective-agents) for the agent-vs-workflow framing.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **LangGraphDeep Dive**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-11_3.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-11.3.html` - regenerate this notebook whenever the source HTML is updated.*
